In [ ]:
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from torchvision import transforms, models, datasets
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = Path(r"C:\Users\harsh\OneDrive\Desktop\plant class\without background class")
if not data_dir.exists():
    data_dir = Path(r"D:\plant class\without background class")
if not data_dir.exists():
    raise FileNotFoundError(f"Dataset folder not found: {data_dir}")

class_names = datasets.ImageFolder(str(data_dir)).classes
num_classes = len(class_names)

model_path = Path(r"D:\plant class\efficientnet_plants.pth")
if not model_path.exists():
    model_path = Path(r"C:\Users\harsh\OneDrive\Desktop\plant class\efficientnet_plants.pth")
if not model_path.exists():
    raise FileNotFoundError(f"Model file not found: {model_path}")

model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
state = torch.load(model_path, map_location=device)

if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

if isinstance(state, dict) and "classifier.1.weight" in state:
    checkpoint_classes = state["classifier.1.weight"].shape[0]
    if checkpoint_classes != num_classes:
        print(
            f"Warning: checkpoint has {checkpoint_classes} classes but dataset has {num_classes}. "
            "Replacing the classifier head to match the current dataset."
        )
        state = {k: v for k, v in state.items() if not k.startswith("classifier.1")}
        model.load_state_dict(state, strict=False)
    else:
        model.load_state_dict(state)
else:
    model.load_state_dict(state, strict=False)

model.to(device)
model.eval()

img_size = (256, 256)
inference_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

def predict_image(img_path):
    img_path = Path(img_path)
    if not img_path.is_file():
        raise FileNotFoundError(f"Image not found: {img_path}")

    image = Image.open(img_path).convert("RGB")
    tensor = inference_transform(image).unsqueeze(0).to(device)

    with torch.inference_mode():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

    idx = int(np.argmax(probs))
    confidence = float(probs[idx])
    print(f"Prediction: {class_names[idx]}")
    print(f"Confidence: {confidence * 100:.2f}%")
    return {
        "prediction": class_names[idx],
        "confidence": confidence,
        "all_scores": {class_names[i]: float(probs[i]) for i in range(len(class_names))},
    }


OSError: [WinError 4551] An Application Control policy has blocked this file. Error loading "c:\Users\harsh\anaconda3\envs\gpu_env\lib\site-packages\torch\lib\shm.dll" or one of its dependencies.

In [6]:
result = predict_image(r"D:\plant class\128.jpg")
print(result)


Prediction: Insulin
Confidence: 99.99%
{'prediction': 'Insulin', 'confidence': 0.9999344348907471, 'all_scores': {'Amrathaballi': 5.7586767070461065e-06, 'Arive Dantu': 4.578652692543983e-07, 'Ashoka': 2.305505546473796e-07, 'Asthma Weed': 3.6317976537247887e-07, 'Balloon Vine': 5.01939950936503e-07, 'Betel': 4.26658516516909e-07, 'Bhrami': 3.9560673030791804e-05, 'Bringaraja': 7.570341153950721e-07, 'Caricature': 3.8621198683586044e-08, 'Catharanthus': 5.647361263072526e-07, 'Curry': 2.0357029839601637e-08, 'Drumstick': 3.212380761397071e-06, 'Globe Amarnath': 4.213693838295285e-09, 'Hibiscus': 3.1179510529000254e-07, 'Insulin': 0.9999344348907471, 'Kamakasturi': 1.5277531417723367e-07, 'Karanda': 7.15915462023986e-07, 'Kohlrabi': 2.3776842681400012e-06, 'Malabar Nut': 1.0050208487655254e-07, 'Malabar Spinach': 9.822125321079511e-06, 'Mint': 2.6848532286294358e-08, 'Nelavembu': 4.0043204307949054e-08, 'Nooni': 4.448368784437662e-08, 'Padri': 2.3108037705554807e-09, 'Parijatha': 1.7606